# Selection XGB

# Setup

## Imports

In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis_project import config as global_config

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
logger = logging.getLogger(__name__)

## Utilities

# Main

## Common

## Load Data

In [ ]:
import pathlib

import yaml

from thesis_project.importance_xgb import config, data_paths

In [ ]:
run = "002"
dir = f"run_{run}"
base_path = global_config.INT_DIR / TICKER / "importance-xgb" / "runs"
path = base_path / dir

with open(path / "config.yaml") as f:
    config_dict = yaml.safe_load(f)
exp_config_obj = config.AnalysisConfig(**config_dict)

with open(path / "data_paths.yaml") as f:
    data_paths_dict = yaml.safe_load(f)
    target = next((name, pathlib.Path(path)) for name, path in data_paths_dict["target"].items())
exp_data_paths = data_paths.DataPathsConfig(
    ticker=TICKER, target=target, features=data_paths_dict["features"]
)

with open(path / "metadata.yaml") as f:
    exp_metadata = yaml.safe_load(f)

selection_hist_dict = {}
for seed in exp_metadata["seeds"]:
    selection_hist_dict[seed] = pd.read_csv(path / f"selection_history_seed_{seed}.csv")

ranking_hist_dict = {}
for seed in exp_metadata["seeds"]:
    ranking_hist_dict[seed] = pd.read_excel(path / f"ranking_history_seed_{seed}.xlsx")

In [ ]:
train_rmse_list = []
val_rmse_list = []
min_len_train = np.inf
min_len_val = np.inf
objective = "rmse"

for seed, selection_hist in selection_hist_dict.items():
    train_rmse_list.append(selection_hist[f"train_{objective}"])
    val_rmse_list.append(selection_hist[f"val_{objective}"])
    min_len_train = min(min_len_train, len(selection_hist[f"train_{objective}"]))
    min_len_val = min(min_len_val, len(selection_hist[f"val_{objective}"]))

for i, _ in enumerate(train_rmse_list):
    train_rmse_list[i] = train_rmse_list[i][:min_len_train]

for i, _ in enumerate(val_rmse_list):
    val_rmse_list[i] = val_rmse_list[i][:min_len_val]


train_rmse_mean = np.mean(train_rmse_list, axis=0)
train_rmse_std = np.std(train_rmse_list, axis=0)
val_rmse_mean = np.mean(val_rmse_list, axis=0)
val_rmse_std = np.std(val_rmse_list, axis=0)

fig, ax = plt.subplots(figsize=(8, 6))

iterations = np.arange(min_len_train)
ax.plot(iterations, train_rmse_mean, label=f"train {objective.upper()}")
ax.fill_between(
    iterations,
    train_rmse_mean - train_rmse_std,
    train_rmse_mean + train_rmse_std,
    alpha=0.4,
)

iterations = np.arange(min_len_val)
ax.plot(iterations, val_rmse_mean, label=f"validation {objective.upper()}")
ax.fill_between(
    iterations,
    val_rmse_mean - val_rmse_std,
    val_rmse_mean + val_rmse_std,
    alpha=0.4,
)

ax.set_xlabel("Iteration number")

ax.set_ylabel(f"{objective.upper()}")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="upper left")
ax.set_title(f"Iterative feature selection mean {objective.upper()}", size=12)

fig.tight_layout()
plt.show()

In [ ]:
def selection_gain_mean_ranking(selection_dict, n_features=20):
    dfs = []

    for seed, df in selection_dict.items():
        tmp = df[["selected_feature", "iteration", "gain"]].copy()
        tmp["seed"] = seed
        dfs.append(tmp)

    all_selection = pd.concat(dfs, ignore_index=True)

    summary = (
        all_selection.groupby("selected_feature")
        .agg(
            mean_gain=("gain", "mean"),
            std_gain=("gain", "std"),
            mean_iteration=("iteration", "mean"),
        )
        .fillna(0)
        .sort_values("mean_gain", ascending=False)
        .head(n_features)
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.barh(
        summary["selected_feature"],
        summary["mean_gain"],
        xerr=summary["std_gain"],
        capsize=3,
    )

    ax.invert_yaxis()

    ax.set_xlabel("Mean Gain")
    ax.set_ylabel("Feature")
    ax.set_title(f"Top {n_features} Features by Mean Gain")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(left=0)

    fig.tight_layout()
    plt.show()

In [ ]:
selection_gain_mean_ranking(selection_hist_dict, 25)